In [1]:
# ===================================================================
# ⚙️ 1. SETUP AND CONFIGURATION
# ===================================================================
# Import libraries, define paths, and load configuration files.
# ===================================================================
import pandas as pd
from pathlib import Path
import yaml
import logging

# Import our custom gaitlab modules
from gaitlab.utils import discover_subjects
from gaitlab.core import Trial
from gaitlab.analysis import (
    generate_coverage_report,
    calculate_normative_stats,
    calculate_spatiotemporal_stats,
    generate_kinetic_qc_report
)
from gaitlab.plotting import PngPlotter, HtmlPlotter

# --- Configuration Paths ---
# ⚠️ ACTION: Update these paths to match your system.
BASE_DIR = Path(r"C:/Users/fx517/Documents/codigo/results/EUROBENCH_RESULTS/c3d_CES/VICON")
OUTPUT_DIR = Path("./output")
CONFIG_DIR = Path("./config")

# --- Directory and Logging Setup ---
subfolders = ["figures", "tables", "logs", "html"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for sf in subfolders:
    (OUTPUT_DIR / sf).mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(OUTPUT_DIR / "logs" / "pipeline_notebook.log", mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ]
)

# --- Load Configuration Files ---
try:
    with open(CONFIG_DIR / "variables_map.yaml", "r") as f:
        VARS_MAP = yaml.safe_load(f)
    with open(CONFIG_DIR / "plot_layout.yaml", "r") as f:
        LAYOUT_CONFIG = yaml.safe_load(f)
    logging.info("Configuration files loaded successfully.")
except FileNotFoundError as e:
    logging.error(f"Configuration file not found: {e}.")
    VARS_MAP, LAYOUT_CONFIG = {}, {}

2026-07-02 15:32:32,083 - INFO - Configuration files loaded successfully.


In [2]:
# ===================================================================
# 🚀 2. PIPELINE EXECUTION
# ===================================================================
# Discover subjects and process each trial in a loop.
# ===================================================================
logging.info("--- Starting Pipeline Execution ---")

subjects_list, files_by_subject = discover_subjects(BASE_DIR)
logging.info(f"Discovered {len(subjects_list)} subjects.")

all_timeseries = []
all_spatiotemporal = []
kinetic_qc_records = []

for subject_id in subjects_list:
    try:
        trial = Trial(subject_id, files_by_subject[subject_id], VARS_MAP)
        if trial.is_valid:
            timeseries_data = trial.process_time_series()
            if timeseries_data is not None:
                all_timeseries.append(timeseries_data)

            if trial.spatiotemporal_params:
                all_spatiotemporal.extend(trial.spatiotemporal_params)

            for side, valid in trial.kinetic_valid.items():
                kinetic_qc_records.append((subject_id, side, valid))
    except Exception as e:
        logging.error(f"Critical error processing {subject_id}: {e}", exc_info=True)

# --- Consolidate Results ---
if all_timeseries:
    master_df = pd.concat(all_timeseries, ignore_index=True)
    spatiotemporal_df = pd.DataFrame(all_spatiotemporal)
    logging.info(f"Successfully processed data for {master_df['subject_id'].nunique()} subjects.")
    print("--- Master DataFrame Preview ---")
    display(master_df.head())
else:
    logging.warning("Pipeline finished, but no data was processed.")
    master_df, spatiotemporal_df = pd.DataFrame(), pd.DataFrame()


2026-07-02 15:32:32,090 - INFO - --- Starting Pipeline Execution ---


2026-07-02 15:32:32,143 - INFO - Discovered 181 subjects.


2026-07-02 15:32:32,186 - WARNING - subject_002: kinetic data flagged invalid on left (peak |forces.ankle.z.left|=nan < threshold=0.5)


2026-07-02 15:32:32,187 - WARNING - subject_002: kinetic data flagged invalid on right (peak |forces.ankle.z.right|=nan < threshold=0.5)


2026-07-02 15:32:32,334 - WARNING - subject_005: kinetic data flagged invalid on left (peak |forces.ankle.z.left|=nan < threshold=0.5)


...

2026-07-02 15:32:40,501 - WARNING - subject_176: kinetic data flagged invalid on left (peak |forces.ankle.z.left|=0.1339619934699385 < threshold=0.5)


2026-07-02 15:32:40,502 - WARNING - subject_176: kinetic data flagged invalid on right (peak |forces.ankle.z.right|=0.2390462511564698 < threshold=0.5)


2026-07-02 15:32:40,787 - INFO - Successfully processed data for 181 subjects.


--- Master DataFrame Preview ---


,subject_id,canonical_variable,side,0,1,2,3,4,5,6,...,42,43,44,45,46,47,48,49,50,kinetic_valid
0,subject_001,angles.pelvis.x.left,left,4.602764,4.605204,4.604534,4.605655,4.613816,4.633070,4.667278,...,5.973037,5.945259,5.889366,5.804383,5.689005,5.544058,5.368115,5.161177,4.923437,NaN
1,subject_001,angles.pelvis.x.right,right,4.065441,4.014660,3.959558,3.902478,3.846357,3.794245,3.749565,...,5.517519,5.509962,5.483638,5.441420,5.387988,5.328534,5.268434,5.213201,5.167657,NaN
2,subject_001,angles.pelvis.y.left,left,0.681206,0.856910,1.031504,1.195656,1.339530,1.456038,1.538106,...,-0.323421,-0.197632,-0.077732,0.041810,0.167911,0.306748,0.464362,0.644569,0.849063,NaN
3,subject_001,angles.pelvis.y.right,right,0.085887,0.298349,0.469021,0.598176,0.686085,0.736676,0.752406,...,-0.958634,-0.887767,-0.800937,-0.690194,-0.549976,-0.378501,-0.175731,0.052185,0.296416,NaN
4,subject_001,angles.pelvis.z.left,left,0.799303,1.006567,1.202879,1.387260,1.558564,1.716245,1.858809,...,-1.664048,-1.443630,-1.209548,-0.963032,-0.704247,-0.433457,-0.149615,0.147939,0.459754,NaN


In [3]:
# ===================================================================
# 📊 3. ANALYSIS AND REPORTING
# ===================================================================
# Generate summary tables from the processed data.
# ===================================================================
if not master_df.empty:
    logging.info("--- Starting Analysis and Reporting ---")
    generate_coverage_report(master_df, OUTPUT_DIR)
    normative_stats = calculate_normative_stats(master_df, OUTPUT_DIR)
    calculate_spatiotemporal_stats(spatiotemporal_df, OUTPUT_DIR)
    generate_kinetic_qc_report(kinetic_qc_records, OUTPUT_DIR)
else:
    logging.warning("Master DataFrame is empty. Skipping analysis.")

2026-07-02 15:32:40,805 - INFO - --- Starting Analysis and Reporting ---


2026-07-02 15:32:40,805 - INFO - Calculating variable coverage...


2026-07-02 15:32:40,811 - INFO - Coverage report saved to output\tables\coverage_by_variable.csv



      Top 15 Variables with HIGHEST Coverage


,canonical_variable,subject_count,coverage_pct
0,angles.ankle.x.left,181,100.0
1,angles.ankle.x.right,181,100.0
2,angles.ankle.z.left,181,100.0
3,angles.ankle.z.right,181,100.0
15,angles.hip.y.right,181,100.0
14,angles.hip.y.left,181,100.0
13,angles.hip.x.right,181,100.0
12,angles.hip.x.left,181,100.0
11,angles.footprogress.z.right,181,100.0
10,angles.footprogress.z.left,181,100.0


...

,walking_speed_m_s,cadence_steps_min,stride_length_m,step_length_m,stance_time_s,swing_time_s,stance_pct
count,362.000,362.000,362.000,362.000,362.000,362.000,362.000
mean,1.024,108.451,1.130,0.553,0.689,0.427,61.626
std,0.159,10.017,0.119,0.058,0.081,0.039,2.407
min,0.663,77.720,0.860,0.383,0.530,0.305,51.456
5%,0.777,92.173,0.935,0.457,0.576,0.367,58.244
25%,0.909,101.437,1.050,0.515,0.625,0.403,60.052
50%,1.018,108.303,1.127,0.552,0.684,0.424,61.549
75%,1.138,115.942,1.201,0.586,0.741,0.451,63.058
95%,1.290,124.863,1.334,0.651,0.819,0.490,65.527
max,1.506,133.630,1.539,0.711,1.088,0.557,70.466


2026-07-02 15:32:40,886 - INFO - Spatiotemporal normative statistics saved to output\tables\spatiotemporal_normative_stats.csv


2026-07-02 15:32:40,889 - INFO - Kinetic QC report saved to output\tables\kinetic_qc_flags.csv (56/362 subject-sides flagged invalid)



      Subjects Flagged for Invalid Kinetic Data


,subject_id,side,kinetic_valid
2,subject_002,left,False
3,subject_002,right,False
8,subject_005,left,False
9,subject_005,right,False
16,subject_009,left,False
17,subject_009,right,False
24,subject_013,left,False
25,subject_013,right,False
42,subject_022,left,False
43,subject_022,right,False


In [4]:
# ===================================================================
# 📈 4. VISUALIZATION
# ===================================================================
# Create all static and interactive plots.
# ===================================================================
if not master_df.empty and 'normative_stats' in locals():
    logging.info("--- Starting Visualization ---")

    # Generate static PNGs
    png_plotter = PngPlotter(LAYOUT_CONFIG, OUTPUT_DIR)
    png_plotter.plot_all_panels(master_df, normative_stats)
    print(f"\n✅ All PNG plots saved in '{OUTPUT_DIR / 'figures'}'")

    # Generate interactive HTML reports
    html_plotter = HtmlPlotter(LAYOUT_CONFIG, OUTPUT_DIR)
    html_plotter.plot_all_panels(master_df, normative_stats)
    print(f"✅ All HTML reports saved in '{OUTPUT_DIR / 'html'}'")

    logging.info("--- ✅ Pipeline Finished Successfully! ---")
else:
    logging.warning("Master or Normative DataFrame is empty. Skipping visualization.")

2026-07-02 15:32:40,899 - INFO - --- Starting Visualization ---


2026-07-02 15:32:40,900 - INFO - Generating all PNG plot panels...


2026-07-02 15:32:40,900 - INFO - Generating PNG panel: Joint Rotation Angles...


2026-07-02 15:32:48,381 - INFO - Saved PNG panel to output\figures\joint_rotation_angles.png


2026-07-02 15:32:48,381 - INFO - Generating PNG panel: Joint Moments...


...

2026-07-02 15:33:15,297 - INFO - Saved HTML panel to output\html\joint_forces.html


2026-07-02 15:33:15,297 - INFO - Generating HTML panel: Ground Reaction Forces...


2026-07-02 15:33:16,573 - INFO - Saved HTML panel to output\html\ground_reaction_forces.html


2026-07-02 15:33:16,574 - INFO - --- ✅ Pipeline Finished Successfully! ---


✅ All HTML reports saved in 'output\html'
